# 07: Cross-Band Transfer Analysis

## Overview
This notebook investigates whether **representational similarity** between frequency bands
predicts **cross-band transfer performance** (Phase 1). It bridges representational metrics
(Phase 3) with functional transfer patterns (Phase 1) and structural circuit overlap (Phase 2).

## Key Questions
1. Do bands with more similar embeddings transfer better?
2. Does residual stream similarity at key layers predict transfer?
3. Does structural overlap (Jaccard) correlate with representational similarity?
4. Do convergence layer differences predict transfer?
5. Can representational metrics predict transfer **beyond** Jaccard (incremental R²)?
6. Is there an asymmetry: LF->HF vs HF->LF in representational space?

## Data Sources
- **Phase 1**: `full_transfer_data.csv` (transfer accuracy: model x train_band x test_band x draw)
- **Phase 2**: `band_jaccard.csv` (circuit overlap: Jaccard similarity)
- **NB01**: `01_cka_matrices.csv` (embedding CKA between bands)
- **NB02**: `02_rsa_trajectory.csv`, `02_probe_trajectory.csv`, `02_separation_trajectory.csv`
- **NB03**: `03_convergence_layers.csv` (logit lens convergence)
- **NB04**: `04_head_role_stability.csv` (attention head role stability)
- **NB06**: `06_mi_ksg_trajectory.csv` (information-theoretic MI)

## Sections
1. Data Loading & Integration
2. Embedding CKA vs Transfer
3. Residual Stream Similarity vs Transfer
4. Structural Overlap vs Representational Similarity (Mantel test)
5. Convergence Layer vs Transfer
6. Predictive Model: Incremental R²
7. Asymmetry Analysis: LF->HF vs HF->LF
8. Summary

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from functools import partial as _partial
from itertools import combinations

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    FREQUENCY_BANDS,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    RANDOM_SEED,
    ALPHA,
    N_BOOTSTRAP,
    LOW_FREQ_BANDS,
    HIGH_FREQ_BANDS,
    ADJACENT_PAIRS,
    CROSS_SPECTRUM_PAIRS,
    get_domain_dirs,
)
from utils.data_loading import (
    save_analysis,
    load_domain_csv,
    load_functional_data,
    load_jaccard_matrices,
    load_phase_csv,
)
from utils.plotting import setup_plotting, save_figure

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

setup_plotting()
np.random.seed(RANDOM_SEED)

# Output directories
ANALYSIS_DIR, VIZ_DIR = get_domain_dirs("transfer", "transfer")
save_analysis = _partial(save_analysis, analysis_dir=ANALYSIS_DIR)
save_figure = _partial(save_figure, viz_dir=VIZ_DIR)

print(f"Transfer analysis: {ANALYSIS_DIR}")
print(f"Transfer viz: {VIZ_DIR}")

Transfer analysis: LSC_circuit_analysis/03_Phase_Representational/outputs/transfer/analysis
Transfer viz: LSC_circuit_analysis/03_Phase_Representational/outputs/transfer/viz


## 1. Data Loading & Integration

Load transfer data (Phase 1), Jaccard similarities (Phase 2), and representational
metrics (Phase 3 NB01-NB06). Merge into a unified pairwise-band DataFrame.

In [2]:
# --- Phase 1: Transfer data ---
_, df_transfer_raw = load_functional_data()
print(f"Transfer data: {len(df_transfer_raw)} rows")
print(f"Columns: {list(df_transfer_raw.columns)}")

# Keep cross-band transfer only (train_band != test_band)
df_transfer = df_transfer_raw[df_transfer_raw["same_band"] == False].copy()
# Also keep same-band as baseline
df_same = df_transfer_raw[df_transfer_raw["same_band"] == True].copy()
print(f"Cross-band pairs: {len(df_transfer)}, Same-band: {len(df_same)}")

Transfer data: 375 rows
Columns: ['model', 'train_band', 'draw', 'test_band', 'same_band', 'base_accuracy', 'base_top5_accuracy', 'base_top10_accuracy', 'base_mean_correct_prob', 'circuit_accuracy', 'circuit_top5_accuracy', 'circuit_top10_accuracy', 'circuit_mean_correct_prob', 'circuit_kl_div', 'train_rank', 'test_rank', 'freq_distance', 'retention_ratio']
Cross-band pairs: 300, Same-band: 75


In [3]:
# --- Phase 2: Jaccard similarities ---
df_jaccard = load_jaccard_matrices()
print(f"Jaccard data: {len(df_jaccard)} rows")
print(df_jaccard.head())

Jaccard data: 125 rows
        model band_1     band_2  mean_jaccard
0  pythia-70m    low        low      0.810724
1  pythia-70m    low     medium      0.790890
2  pythia-70m    low       high      0.770530
3  pythia-70m    low  very_high      0.724777
4  pythia-70m    low    control      0.726767


In [4]:
# --- Phase 3: Representational metrics ---

# NB01: Embedding CKA (pairwise band similarity)
try:
    df_emb_cka = load_domain_csv("embedding", "base", "01_cka_matrices.csv")
    print(f"Embedding CKA: {len(df_emb_cka)} rows")
except FileNotFoundError:
    df_emb_cka = pd.DataFrame()
    print("Embedding CKA not found")

# NB02: Probe trajectory, RSA, Separation
try:
    df_probe = load_domain_csv("residual_stream", "base", "02_probe_trajectory.csv")
    print(f"Probe trajectory: {len(df_probe)} rows")
except FileNotFoundError:
    df_probe = pd.DataFrame()
    print("Probe trajectory not found")

try:
    df_rsa = load_domain_csv("residual_stream", "base", "02_rsa_trajectory.csv")
    print(f"RSA trajectory: {len(df_rsa)} rows")
except FileNotFoundError:
    df_rsa = pd.DataFrame()
    print("RSA trajectory not found")

try:
    df_sep = load_domain_csv("residual_stream", "base", "02_separation_trajectory.csv")
    print(f"Separation trajectory: {len(df_sep)} rows")
except FileNotFoundError:
    df_sep = pd.DataFrame()
    print("Separation trajectory not found")

# NB03: Convergence layers
try:
    df_conv = load_domain_csv("logit_lens", "base", "03_convergence_layers.csv")
    print(f"Convergence layers: {len(df_conv)} rows")
except FileNotFoundError:
    df_conv = pd.DataFrame()
    print("Convergence layers not found")

# NB06: MI trajectory
try:
    df_mi = load_domain_csv("info_theoretic", "base", "06_mi_ksg_trajectory.csv")
    print(f"MI trajectory: {len(df_mi)} rows")
except FileNotFoundError:
    df_mi = pd.DataFrame()
    print("MI trajectory not found")

Embedding CKA: 120 rows
Probe trajectory: 174 rows
RSA trajectory: 58 rows
Separation trajectory: 174 rows
Convergence layers: 75 rows
MI trajectory: 174 rows


In [5]:
# Build unified pairwise-band DataFrame
# Key: (model, band_1, band_2): averaging over draws for representational metrics
# Use FREQUENCY_BANDS (excludes 'control' which has FREQUENCY_RANK=None)

rows = []
for model in MODELS:
    for b1, b2 in combinations(BANDS, 2):
        row = {"model": model, "band_1": b1, "band_2": b2}
        row["model_capacity"] = MODEL_CAPACITY.get(model, 0)
        r1 = FREQUENCY_RANK.get(b1)
        r2 = FREQUENCY_RANK.get(b2)
        row["freq_distance"] = (
            abs(r1 - r2) if (r1 is not None and r2 is not None) else np.nan
        )

        # Transfer accuracy (directional: b1->b2 and b2->b1)
        t_12 = df_transfer[
            (df_transfer["model"] == model)
            & (df_transfer["train_band"] == b1)
            & (df_transfer["test_band"] == b2)
        ]["circuit_accuracy"]
        t_21 = df_transfer[
            (df_transfer["model"] == model)
            & (df_transfer["train_band"] == b2)
            & (df_transfer["test_band"] == b1)
        ]["circuit_accuracy"]

        row["transfer_12"] = t_12.mean() if len(t_12) > 0 else np.nan
        row["transfer_21"] = t_21.mean() if len(t_21) > 0 else np.nan
        row["transfer_mean"] = np.nanmean([row["transfer_12"], row["transfer_21"]])
        row["transfer_asymmetry"] = row["transfer_12"] - row["transfer_21"]

        # Jaccard (symmetric)
        jac = df_jaccard[
            (df_jaccard["model"] == model)
            & (
                ((df_jaccard["band_1"] == b1) & (df_jaccard["band_2"] == b2))
                | ((df_jaccard["band_1"] == b2) & (df_jaccard["band_2"] == b1))
            )
        ]["mean_jaccard"]
        row["jaccard"] = jac.values[0] if len(jac) > 0 else np.nan

        # Embedding CKA (symmetric)
        if not df_emb_cka.empty:
            cka = df_emb_cka[
                (df_emb_cka["model"] == model)
                & (
                    ((df_emb_cka["band_1"] == b1) & (df_emb_cka["band_2"] == b2))
                    | ((df_emb_cka["band_1"] == b2) & (df_emb_cka["band_2"] == b1))
                )
            ]["cka"]
            row["embedding_cka"] = cka.mean() if len(cka) > 0 else np.nan

        rows.append(row)

df_pairs = pd.DataFrame(rows)
print(f"Pairwise pairs: {len(df_pairs)} rows")
print(f"Columns: {list(df_pairs.columns)}")
df_pairs.head()

Pairwise pairs: 50 rows
Columns: ['model', 'band_1', 'band_2', 'model_capacity', 'freq_distance', 'transfer_12', 'transfer_21', 'transfer_mean', 'transfer_asymmetry', 'jaccard', 'embedding_cka']


,model,band_1,band_2,model_capacity,freq_distance,transfer_12,transfer_21,transfer_mean,transfer_asymmetry,jaccard,embedding_cka
0,pythia-70m,low,medium,70,1.0,0.322963,0.251852,0.287407,0.071111,0.790890,0.474499
1,pythia-70m,low,high,70,2.0,0.358519,0.269630,0.314074,0.088889,0.770530,0.469160
2,pythia-70m,low,very_high,70,3.0,0.468148,0.231111,0.349630,0.237037,0.724777,0.470316
3,pythia-70m,low,control,70,NaN,0.432593,0.262222,0.347407,0.170370,0.726767,0.486547
4,pythia-70m,medium,high,70,1.0,0.388148,0.345185,0.366667,0.042963,0.778754,0.463691


In [6]:
# Add residual-stream metrics to pairwise dataframe
# For each model x band pair, compute:
#   - probe_diff: difference in max probe accuracy between bands
#   - convergence_diff: difference in mean convergence layer

# Probe: max accuracy across layers per model/band (averaged over draws)
if not df_probe.empty:
    probe_max = df_probe.groupby(["model", "band"])["accuracy"].max().reset_index()
    probe_max.rename(columns={"accuracy": "max_probe_acc"}, inplace=True)

    for idx, row in df_pairs.iterrows():
        m, b1, b2 = row["model"], row["band_1"], row["band_2"]
        p1 = probe_max[(probe_max["model"] == m) & (probe_max["band"] == b1)][
            "max_probe_acc"
        ]
        p2 = probe_max[(probe_max["model"] == m) & (probe_max["band"] == b2)][
            "max_probe_acc"
        ]
        if len(p1) > 0 and len(p2) > 0:
            df_pairs.loc[idx, "probe_diff"] = abs(p1.values[0] - p2.values[0])

# Convergence: mean convergence layer per model/band
if not df_conv.empty:
    conv_mean = (
        df_conv.groupby(["model", "band"])["mean_convergence_layer"]
        .mean()
        .reset_index()
    )

    for idx, row in df_pairs.iterrows():
        m, b1, b2 = row["model"], row["band_1"], row["band_2"]
        c1 = conv_mean[(conv_mean["model"] == m) & (conv_mean["band"] == b1)][
            "mean_convergence_layer"
        ]
        c2 = conv_mean[(conv_mean["model"] == m) & (conv_mean["band"] == b2)][
            "mean_convergence_layer"
        ]
        if len(c1) > 0 and len(c2) > 0:
            df_pairs.loc[idx, "convergence_diff"] = abs(c1.values[0] - c2.values[0])

# Separation ratio: max separation per model (not per band: it's computed across bands)
# Instead compute per-model peak separation as a control
if not df_sep.empty:
    sep_peak = df_sep.groupby("model")["separation_ratio"].max().reset_index()
    sep_peak.rename(columns={"separation_ratio": "peak_separation"}, inplace=True)
    df_pairs = df_pairs.merge(sep_peak, on="model", how="left")

save_analysis(df_pairs, "07_pairwise_transfer_metrics.csv")
print(f"Final pairwise: {len(df_pairs)} rows, {len(df_pairs.columns)} columns")
df_pairs.describe()

Final pairwise: 50 rows, 13 columns


,model_capacity,freq_distance,transfer_12,transfer_21,transfer_mean,transfer_asymmetry,jaccard,embedding_cka,convergence_diff,peak_separation
count,50.000000,30.000000,50.000000,50.000000,50.000000,50.000000,50.000000,40.000000,50.000000,40.000000
mean,608.000000,1.666667,0.834400,0.765837,0.800119,0.068563,0.516200,0.563710,1.374756,14.815223
std,517.111286,0.758098,0.199738,0.230250,0.211987,0.077885,0.140217,0.063544,1.638822,1.167918
min,70.000000,1.000000,0.322963,0.231111,0.287407,-0.080000,0.335544,0.463691,0.017778,13.262400
25%,160.000000,1.000000,0.905926,0.718148,0.821296,0.015556,0.423910,0.522703,0.277037,14.103081
50%,410.000000,1.500000,0.921481,0.867407,0.900741,0.066667,0.462167,0.566447,0.683704,14.788789
75%,1000.000000,2.000000,0.940000,0.920000,0.928333,0.086667,0.564866,0.607718,2.021852,15.500931
max,1400.000000,3.000000,0.980741,0.968889,0.973333,0.333333,0.790890,0.670338,6.952593,16.420913


## 2. Embedding CKA vs Transfer

Do bands with more similar embeddings (higher CKA in embedding space) show
better cross-band transfer? This tests the simplest representational hypothesis:
transfer is predicted by input similarity.

In [7]:
# Correlation: embedding CKA vs mean transfer accuracy
if "embedding_cka" in df_pairs.columns:
    mask = df_pairs[["embedding_cka", "transfer_mean"]].notna().all(axis=1)
    df_valid = df_pairs[mask]

    r, p = stats.pearsonr(df_valid["embedding_cka"], df_valid["transfer_mean"])
    rho, p_s = stats.spearmanr(df_valid["embedding_cka"], df_valid["transfer_mean"])

    print(f"Embedding CKA vs Transfer (Pearson):  r = {r:.4f}, p = {p:.4e}")
    print(f"Embedding CKA vs Transfer (Spearman): rho = {rho:.4f}, p = {p_s:.4e}")
    print(f"N pairs = {len(df_valid)}")
else:
    print("Embedding CKA not available")

Embedding CKA vs Transfer (Pearson):  r = 0.7903, p = 1.3175e-09
Embedding CKA vs Transfer (Spearman): rho = 0.5668, p = 1.3723e-04
N pairs = 40


In [8]:
if "embedding_cka" in df_pairs.columns:
    mask = df_pairs[["embedding_cka", "transfer_mean"]].notna().all(axis=1)
    df_valid = df_pairs[mask]

    fig, ax = plt.subplots(figsize=(8, 6))
    for model in MODELS:
        df_m = df_valid[df_valid["model"] == model]
        ax.scatter(
            df_m["embedding_cka"],
            df_m["transfer_mean"],
            color=MODEL_COLORS.get(model, "gray"),
            label=model,
            s=50,
            alpha=0.7,
        )

    # Overall regression line
    x = df_valid["embedding_cka"].values
    y = df_valid["transfer_mean"].values
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m * x_line + b, "k--", alpha=0.5, label=f"r={r:.3f}")

    ax.set_xlabel("Embedding CKA")
    ax.set_ylabel("Mean Transfer Accuracy (Circuit)")
    ax.set_title("Embedding Similarity vs Cross-Band Transfer")
    ax.legend()
    fig.tight_layout()
    save_figure(fig, "viz_07_01_embedding_cka_vs_transfer.png")

## 3. Residual Stream Similarity vs Transfer

Compute CKA between residual stream representations of different bands at
key layers (early, middle, final) and correlate with transfer.

In [9]:
from utils.data_loading import load_extracted_activations
from utils.geometry import linear_cka
from utils.constants import MODEL_INFO

# Compute residual CKA between band pairs at early/mid/final layers
rows_resid_cka = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    key_layers = {
        "early": 1,
        "middle": n_layers // 2,
        "final": n_layers - 1,
    }

    for draw in DRAWS:
        # Load all bands
        band_resid = {}
        for band in BANDS:
            try:
                data = load_extracted_activations(model, band, draw)
                band_resid[band] = data["resid_post_predpos"]  # (N, n_layers, d_model)
            except FileNotFoundError:
                pass

        if len(band_resid) < 2:
            continue

        for b1, b2 in combinations(BANDS, 2):
            if b1 not in band_resid or b2 not in band_resid:
                continue

            for layer_name, layer_idx in key_layers.items():
                X1 = band_resid[b1][:, layer_idx, :]
                X2 = band_resid[b2][:, layer_idx, :]
                # Align sample sizes
                n = min(len(X1), len(X2))
                cka_val = linear_cka(X1[:n], X2[:n])

                rows_resid_cka.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band_1": b1,
                        "band_2": b2,
                        "layer_name": layer_name,
                        "layer_idx": layer_idx,
                        "cka": cka_val,
                    }
                )

df_resid_cka = pd.DataFrame(rows_resid_cka)
save_analysis(df_resid_cka, "07_residual_cka_pairwise.csv")
print(f"Residual CKA pairs: {len(df_resid_cka)} rows")
df_resid_cka.head()

Residual CKA pairs: 450 rows


,model,draw,band_1,band_2,layer_name,layer_idx,cka
0,pythia-70m,draw_1,low,medium,early,1,0.129413
1,pythia-70m,draw_1,low,medium,middle,3,0.010905
2,pythia-70m,draw_1,low,medium,final,5,0.025146
3,pythia-70m,draw_1,low,high,early,1,0.161473
4,pythia-70m,draw_1,low,high,middle,3,0.029126


In [10]:
# Merge residual CKA into pairwise df (pivot by layer_name)
if not df_resid_cka.empty:
    resid_cka_avg = (
        df_resid_cka.groupby(["model", "band_1", "band_2", "layer_name"])["cka"]
        .mean()
        .reset_index()
    )

    resid_pivot = resid_cka_avg.pivot_table(
        index=["model", "band_1", "band_2"], columns="layer_name", values="cka"
    ).reset_index()
    resid_pivot.columns = [
        "model",
        "band_1",
        "band_2",
        "resid_cka_early",
        "resid_cka_final",
        "resid_cka_middle",
    ]

    df_pairs = df_pairs.merge(resid_pivot, on=["model", "band_1", "band_2"], how="left")
    print(f"Added residual CKA columns. Pairs now: {len(df_pairs.columns)} cols")

Added residual CKA columns. Pairs now: 16 cols


In [11]:
# Correlations: residual CKA at each layer vs transfer
results_corr = []
for layer_col in ["resid_cka_early", "resid_cka_middle", "resid_cka_final"]:
    if layer_col not in df_pairs.columns:
        continue
    mask = df_pairs[[layer_col, "transfer_mean"]].notna().all(axis=1)
    df_v = df_pairs[mask]
    if len(df_v) < 5:
        continue
    r, p = stats.pearsonr(df_v[layer_col], df_v["transfer_mean"])
    rho, p_s = stats.spearmanr(df_v[layer_col], df_v["transfer_mean"])
    results_corr.append(
        {
            "metric": layer_col,
            "pearson_r": r,
            "pearson_p": p,
            "spearman_rho": rho,
            "spearman_p": p_s,
            "n": len(df_v),
        }
    )
    print(f"{layer_col}: r={r:.4f} (p={p:.4e}), rho={rho:.4f} (p={p_s:.4e})")

df_resid_corr = pd.DataFrame(results_corr)
if not df_resid_corr.empty:
    save_analysis(df_resid_corr, "07_residual_cka_transfer_correlations.csv")

resid_cka_early: r=0.8950 (p=1.8903e-18), rho=0.4114 (p=2.9978e-03)
resid_cka_middle: r=0.8063 (p=1.5988e-12), rho=0.4466 (p=1.1503e-03)
resid_cka_final: r=0.6805 (p=5.4284e-08), rho=0.3136 (p=2.6576e-02)


In [12]:
# Scatter: residual CKA at final layer vs transfer
if "resid_cka_final" in df_pairs.columns:
    mask = df_pairs[["resid_cka_final", "transfer_mean"]].notna().all(axis=1)
    df_v = df_pairs[mask]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, col, label in zip(
        axes,
        ["resid_cka_early", "resid_cka_middle", "resid_cka_final"],
        ["Early Layer", "Middle Layer", "Final Layer"],
    ):
        if col not in df_pairs.columns:
            continue
        mask_c = df_pairs[[col, "transfer_mean"]].notna().all(axis=1)
        df_c = df_pairs[mask_c]

        for model in MODELS:
            df_m = df_c[df_c["model"] == model]
            ax.scatter(
                df_m[col],
                df_m["transfer_mean"],
                color=MODEL_COLORS.get(model, "gray"),
                label=model,
                s=40,
                alpha=0.7,
            )

        r, _ = stats.pearsonr(df_c[col], df_c["transfer_mean"])
        ax.set_xlabel(f"Residual CKA ({label})")
        ax.set_ylabel("Mean Transfer Accuracy")
        ax.set_title(f"{label} (r={r:.3f})")

    axes[0].legend(fontsize=8)
    fig.suptitle("Residual Stream CKA vs Transfer Accuracy", y=1.02)
    fig.tight_layout()
    save_figure(fig, "viz_07_02_residual_cka_vs_transfer.png")

## 4. Structural Overlap vs Representational Similarity

Do circuits that are structurally more similar (higher Jaccard) also produce
more similar representations? Test with Mantel test (correlation between
distance matrices) per model.

In [13]:
def mantel_test(dist1, dist2, n_perms=9999):
    """Mantel test: correlation between two distance/similarity matrices.

    Uses upper triangle only. Returns Pearson r and permutation p-value.
    """
    n = dist1.shape[0]
    idx = np.triu_indices(n, k=1)
    x = dist1[idx]
    y = dist2[idx]

    r_obs = np.corrcoef(x, y)[0, 1]

    count = 0
    for _ in range(n_perms):
        perm = np.random.permutation(n)
        d1_perm = dist1[np.ix_(perm, perm)]
        x_perm = d1_perm[idx]
        r_perm = np.corrcoef(x_perm, y)[0, 1]
        if r_perm >= r_obs:
            count += 1

    p_value = (count + 1) / (n_perms + 1)
    return r_obs, p_value


# Build Jaccard matrix and representation similarity matrix per model
mantel_results = []

for model in MODELS:
    # Jaccard matrix
    n_bands = len(BANDS)
    jac_mat = np.ones((n_bands, n_bands))
    for i, b1 in enumerate(BANDS):
        for j, b2 in enumerate(BANDS):
            if i == j:
                continue
            jac_row = df_jaccard[
                (df_jaccard["model"] == model)
                & (
                    ((df_jaccard["band_1"] == b1) & (df_jaccard["band_2"] == b2))
                    | ((df_jaccard["band_1"] == b2) & (df_jaccard["band_2"] == b1))
                )
            ]["mean_jaccard"]
            if len(jac_row) > 0:
                jac_mat[i, j] = jac_row.values[0]

    # Embedding CKA matrix
    if not df_emb_cka.empty:
        cka_mat = np.ones((n_bands, n_bands))
        for i, b1 in enumerate(BANDS):
            for j, b2 in enumerate(BANDS):
                if i == j:
                    continue
                cka_row = df_emb_cka[
                    (df_emb_cka["model"] == model)
                    & (
                        ((df_emb_cka["band_1"] == b1) & (df_emb_cka["band_2"] == b2))
                        | ((df_emb_cka["band_1"] == b2) & (df_emb_cka["band_2"] == b1))
                    )
                ]["cka"]
                if len(cka_row) > 0:
                    cka_mat[i, j] = cka_row.mean()

        r_jac_cka, p_jac_cka = mantel_test(jac_mat, cka_mat)
        mantel_results.append(
            {
                "model": model,
                "comparison": "jaccard_vs_embedding_cka",
                "mantel_r": r_jac_cka,
                "mantel_p": p_jac_cka,
            }
        )
        print(
            f"{model}: Jaccard vs Embedding CKA: r={r_jac_cka:.4f}, p={p_jac_cka:.4f}"
        )

    # Residual CKA at final layer
    if "resid_cka_final" in df_pairs.columns:
        resid_mat = np.ones((n_bands, n_bands))
        for i, b1 in enumerate(BANDS):
            for j, b2 in enumerate(BANDS):
                if i == j:
                    continue
                r_val = df_pairs[
                    (df_pairs["model"] == model)
                    & (
                        ((df_pairs["band_1"] == b1) & (df_pairs["band_2"] == b2))
                        | ((df_pairs["band_1"] == b2) & (df_pairs["band_2"] == b1))
                    )
                ]["resid_cka_final"]
                if len(r_val) > 0:
                    resid_mat[i, j] = r_val.values[0]

        r_jac_resid, p_jac_resid = mantel_test(jac_mat, resid_mat)
        mantel_results.append(
            {
                "model": model,
                "comparison": "jaccard_vs_resid_cka_final",
                "mantel_r": r_jac_resid,
                "mantel_p": p_jac_resid,
            }
        )
        print(
            f"{model}: Jaccard vs Resid CKA (final): r={r_jac_resid:.4f}, p={p_jac_resid:.4f}"
        )

df_mantel = pd.DataFrame(mantel_results)
if not df_mantel.empty:
    save_analysis(df_mantel, "07_mantel_test_results.csv")
    print(f"\nMantel results saved: {len(df_mantel)} tests")

pythia-70m: Jaccard vs Embedding CKA: r=-0.4116, p=0.9319


pythia-70m: Jaccard vs Resid CKA (final): r=0.1980, p=0.2874


pythia-160m: Jaccard vs Embedding CKA: r=0.1113, p=0.3719


pythia-160m: Jaccard vs Resid CKA (final): r=-0.5003, p=0.9638


pythia-410m: Jaccard vs Embedding CKA: r=-0.0935, p=0.6333


...

pythia-1b: Jaccard vs Embedding CKA: r=0.3857, p=0.1802


pythia-1b: Jaccard vs Resid CKA (final): r=-0.2588, p=0.7487


<TMPDIR>/env/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
<TMPDIR>/env/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


pythia-1.4b: Jaccard vs Embedding CKA: r=nan, p=0.0001


pythia-1.4b: Jaccard vs Resid CKA (final): r=0.0981, p=0.4268

Mantel results saved: 10 tests


In [14]:
# Scatter: Jaccard vs Embedding CKA (per model)
if "embedding_cka" in df_pairs.columns and "jaccard" in df_pairs.columns:
    mask = df_pairs[["jaccard", "embedding_cka"]].notna().all(axis=1)
    df_v = df_pairs[mask]

    fig, ax = plt.subplots(figsize=(8, 6))
    for model in MODELS:
        df_m = df_v[df_v["model"] == model]
        ax.scatter(
            df_m["jaccard"],
            df_m["embedding_cka"],
            color=MODEL_COLORS.get(model, "gray"),
            label=model,
            s=50,
            alpha=0.7,
        )

    r, p = stats.pearsonr(df_v["jaccard"], df_v["embedding_cka"])
    ax.set_xlabel("Circuit Jaccard Similarity")
    ax.set_ylabel("Embedding CKA")
    ax.set_title(f"Structural vs Representational Similarity (r={r:.3f}, p={p:.4e})")
    ax.legend()
    fig.tight_layout()
    save_figure(fig, "viz_07_03_jaccard_vs_embedding_cka.png")

## 5. Convergence Layer vs Transfer

Do band pairs with similar convergence layers (from logit lens) transfer better?
Large convergence differences may indicate different computational strategies.

In [15]:
if "convergence_diff" in df_pairs.columns:
    mask = df_pairs[["convergence_diff", "transfer_mean"]].notna().all(axis=1)
    df_v = df_pairs[mask]

    if len(df_v) >= 5:
        r, p = stats.pearsonr(df_v["convergence_diff"], df_v["transfer_mean"])
        rho, p_s = stats.spearmanr(df_v["convergence_diff"], df_v["transfer_mean"])
        print(
            f"Convergence diff vs Transfer: r={r:.4f} (p={p:.4e}), rho={rho:.4f} (p={p_s:.4e})"
        )
        print(f"N = {len(df_v)}")
    else:
        print("Not enough data points for convergence analysis")
else:
    print("Convergence diff not available")

Convergence diff vs Transfer: r=0.2285 (p=1.1051e-01), rho=0.1648 (p=2.5283e-01)
N = 50


In [16]:
if "convergence_diff" in df_pairs.columns:
    mask = df_pairs[["convergence_diff", "transfer_mean"]].notna().all(axis=1)
    df_v = df_pairs[mask]

    if len(df_v) >= 5:
        fig, ax = plt.subplots(figsize=(8, 6))
        for model in MODELS:
            df_m = df_v[df_v["model"] == model]
            ax.scatter(
                df_m["convergence_diff"],
                df_m["transfer_mean"],
                color=MODEL_COLORS.get(model, "gray"),
                label=model,
                s=50,
                alpha=0.7,
            )

        ax.set_xlabel("|Convergence Layer Difference|")
        ax.set_ylabel("Mean Transfer Accuracy")
        ax.set_title("Convergence Layer Difference vs Transfer")
        ax.legend()
        fig.tight_layout()
        save_figure(fig, "viz_07_04_convergence_diff_vs_transfer.png")

## 6. Predictive Model: Incremental R²

Can representational metrics predict transfer **beyond** structural overlap (Jaccard)?
Fit OLS: transfer ~ Jaccard, then transfer ~ Jaccard + repr_metrics.
Report incremental R² and partial F-test.

In [17]:
# Identify available predictor columns
repr_cols = [
    c
    for c in [
        "embedding_cka",
        "resid_cka_early",
        "resid_cka_middle",
        "resid_cka_final",
        "convergence_diff",
        "probe_diff",
        "freq_distance",
    ]
    if c in df_pairs.columns
]

base_cols = ["jaccard"]
all_cols = base_cols + repr_cols

# Drop rows with any NaN in relevant columns
mask = df_pairs[all_cols + ["transfer_mean"]].notna().all(axis=1)
df_reg = df_pairs[mask].copy()
print(f"Regression sample: {len(df_reg)} rows")
print(f"Base predictors: {base_cols}")
print(f"Repr predictors: {repr_cols}")
print(f"Total predictors: {len(all_cols)} (need N >> p to avoid overfitting)")

if len(df_reg) >= 10:
    from sklearn.model_selection import LeaveOneOut, cross_val_predict
    from sklearn.metrics import r2_score

    y = df_reg["transfer_mean"].values

    # Model 1: Jaccard only
    X_base = df_reg[base_cols].values
    scaler_base = StandardScaler()
    X_base_s = scaler_base.fit_transform(X_base)
    reg_base = LinearRegression().fit(X_base_s, y)
    r2_base = reg_base.score(X_base_s, y)

    # Model 2: Jaccard + representational
    X_full = df_reg[all_cols].values
    scaler_full = StandardScaler()
    X_full_s = scaler_full.fit_transform(X_full)
    reg_full = LinearRegression().fit(X_full_s, y)
    r2_full = reg_full.score(X_full_s, y)

    delta_r2 = r2_full - r2_base

    # Adjusted R² to penalize predictor count
    n = len(y)
    p_base = X_base_s.shape[1]
    p_full = X_full_s.shape[1]
    adj_r2_base = 1 - (1 - r2_base) * (n - 1) / (n - p_base - 1)
    adj_r2_full = 1 - (1 - r2_full) * (n - 1) / (n - p_full - 1)

    # LOO cross-validated R²: collect all LOO predictions, then compute R² globally
    # (sklearn's cross_val_score with LOO gives NaN because R² is undefined for 1 sample)
    loo = LeaveOneOut()
    loo_pred_base = cross_val_predict(LinearRegression(), X_base_s, y, cv=loo)
    loo_pred_full = cross_val_predict(LinearRegression(), X_full_s, y, cv=loo)
    cv_r2_base = r2_score(y, loo_pred_base)
    cv_r2_full = r2_score(y, loo_pred_full)

    # Partial F-test
    df_num = p_full - p_base
    df_den = n - p_full - 1

    if df_den > 0 and (1 - r2_full) > 0:
        f_stat = (delta_r2 / df_num) / ((1 - r2_full) / df_den)
        p_value = 1 - stats.f.cdf(f_stat, df_num, df_den)
    else:
        f_stat, p_value = np.nan, np.nan

    print(f"\n{'Metric':<30s} {'Jaccard only':>14s} {'Jaccard+Repr':>14s}")
    print(f"{'-' * 60}")
    print(f"{'Training R²':<30s} {r2_base:>14.4f} {r2_full:>14.4f}")
    print(f"{'Adjusted R²':<30s} {adj_r2_base:>14.4f} {adj_r2_full:>14.4f}")
    print(f"{'LOO-CV R²':<30s} {cv_r2_base:>14.4f} {cv_r2_full:>14.4f}")
    print(f"\nIncremental R² (train):  ΔR² = {delta_r2:.4f}")
    print(f"Incremental R² (CV):     ΔR² = {cv_r2_full - cv_r2_base:.4f}")
    print(
        f"Partial F-test:          F({df_num},{df_den}) = {f_stat:.3f}, p = {p_value:.4e}"
    )

    if n < 3 * p_full:
        print(
            f"\nWARNING: N/p ratio = {n / p_full:.1f} (< 3). "
            f"Training R² is inflated. Use CV R² for interpretation."
        )

    # Coefficient table
    coef_df = pd.DataFrame(
        {
            "predictor": all_cols,
            "coefficient": reg_full.coef_,
        }
    ).sort_values("coefficient", key=abs, ascending=False)
    print(f"\nFull model coefficients (standardized):")
    print(coef_df.to_string(index=False))

    # Save results
    r2_results = {
        "r2_base": r2_base,
        "r2_full": r2_full,
        "delta_r2": delta_r2,
        "adj_r2_base": adj_r2_base,
        "adj_r2_full": adj_r2_full,
        "cv_r2_base": cv_r2_base,
        "cv_r2_full": cv_r2_full,
        "delta_r2_cv": cv_r2_full - cv_r2_base,
        "f_stat": f_stat,
        "f_p_value": p_value,
        "n": n,
        "p_base": p_base,
        "p_full": p_full,
    }
    save_analysis(pd.DataFrame([r2_results]), "07_incremental_r2.csv")
    save_analysis(coef_df, "07_regression_coefficients.csv")
else:
    print("Not enough data for regression")

Regression sample: 24 rows
Base predictors: ['jaccard']
Repr predictors: ['embedding_cka', 'resid_cka_early', 'resid_cka_middle', 'resid_cka_final', 'convergence_diff', 'freq_distance']
Total predictors: 7 (need N >> p to avoid overfitting)

Metric                           Jaccard only   Jaccard+Repr
------------------------------------------------------------
Training R²                            0.8600         0.9903
Adjusted R²                            0.8536         0.9861
LOO-CV R²                              0.8356         0.9781

Incremental R² (train):  ΔR² = 0.1304
Incremental R² (CV):     ΔR² = 0.1424
Partial F-test:          F(6,16) = 35.931, p = 2.0581e-08

Full model coefficients (standardized):
       predictor  coefficient
 resid_cka_early     0.351872
         jaccard    -0.101358
   embedding_cka    -0.097234
 resid_cka_final    -0.076868
resid_cka_middle    -0.069964
   freq_distance    -0.026772
convergence_diff     0.007455


In [18]:
# Coefficient plot
if len(df_reg) >= 10:
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["coral" if c in repr_cols else "steelblue" for c in coef_df["predictor"]]
    ax.barh(coef_df["predictor"], coef_df["coefficient"], color=colors)
    ax.axvline(x=0, color="gray", linestyle="-", linewidth=0.5)
    ax.set_xlabel("Standardized Coefficient")
    ax.set_title(f"Transfer Predictors (ΔR² = {delta_r2:.4f}, p = {p_value:.4e})")

    # Legend
    from matplotlib.patches import Patch

    ax.legend(
        handles=[
            Patch(facecolor="steelblue", label="Structural (Jaccard)"),
            Patch(facecolor="coral", label="Representational"),
        ]
    )
    fig.tight_layout()
    save_figure(fig, "viz_07_05_regression_coefficients.png")

In [19]:
# Per-model incremental R² using leave-one-out cross-validation
# Training R² is unreliable here because per-model N is small (6-30 rows)
# relative to predictor count (up to 7). LOO-CV R² avoids overfitting.
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score

rows_model_r2 = []

for model in MODELS:
    df_m = df_reg[df_reg["model"] == model]
    n_m = len(df_m)
    p_full = len(all_cols)

    # Require at least 2x predictors to avoid degenerate fits
    if n_m < max(2 * p_full, 5):
        print(
            f"{model}: SKIP (n={n_m}, need >={max(2 * p_full, 5)} for {p_full} predictors)"
        )
        continue

    y_m = df_m["transfer_mean"].values

    X_b = df_m[base_cols].values
    X_f = df_m[all_cols].values

    # LOO cross-validated R² (collect predictions, compute R² globally)
    loo = LeaveOneOut()
    loo_pred_b = cross_val_predict(LinearRegression(), X_b, y_m, cv=loo)
    loo_pred_f = cross_val_predict(LinearRegression(), X_f, y_m, cv=loo)
    cv_r2_b = r2_score(y_m, loo_pred_b)
    cv_r2_f = r2_score(y_m, loo_pred_f)

    # Also report training R² for comparison (to show overfitting gap)
    r2_train_b = LinearRegression().fit(X_b, y_m).score(X_b, y_m)
    r2_train_f = LinearRegression().fit(X_f, y_m).score(X_f, y_m)

    rows_model_r2.append(
        {
            "model": model,
            "n": n_m,
            "p_full": p_full,
            "r2_jaccard_train": r2_train_b,
            "r2_full_train": r2_train_f,
            "r2_jaccard_cv": cv_r2_b,
            "r2_full_cv": cv_r2_f,
            "delta_r2_cv": cv_r2_f - cv_r2_b,
        }
    )
    print(
        f"{model} (n={n_m}): CV-R²(Jac)={cv_r2_b:.3f}, CV-R²(Full)={cv_r2_f:.3f}, "
        f"ΔR²={cv_r2_f - cv_r2_b:.3f}  [train R²: {r2_train_f:.3f}]"
    )

df_model_r2 = pd.DataFrame(rows_model_r2)
if not df_model_r2.empty:
    save_analysis(df_model_r2, "07_per_model_incremental_r2.csv")
else:
    print("\nNo models had enough data for per-model regression.")

pythia-70m: SKIP (n=6, need >=14 for 7 predictors)
pythia-160m: SKIP (n=6, need >=14 for 7 predictors)
pythia-410m: SKIP (n=6, need >=14 for 7 predictors)
pythia-1b: SKIP (n=6, need >=14 for 7 predictors)
pythia-1.4b: SKIP (n=0, need >=14 for 7 predictors)

No models had enough data for per-model regression.


## 7. Asymmetry Analysis: LF->HF vs HF->LF

Transfer is directional: train on band A, test on band B. Is there a
representational basis for transfer asymmetry (LF->HF != HF->LF)?

In [20]:
# Compute directional transfer with frequency direction
# Only use frequency bands (not control, which has FREQUENCY_RANK=None)
rows_asym = []

for model in MODELS:
    for b1, b2 in combinations(BANDS, 2):
        r1 = FREQUENCY_RANK.get(b1)
        r2 = FREQUENCY_RANK.get(b2)

        # Skip pairs involving control (rank=None)
        if r1 is None or r2 is None:
            continue

        # b1 is always lower rank (lower frequency)
        if r1 > r2:
            b1, b2 = b2, b1
            r1, r2 = r2, r1

        # LF->HF: train on b1 (lower freq), test on b2 (higher freq)
        lf_hf = df_transfer[
            (df_transfer["model"] == model)
            & (df_transfer["train_band"] == b1)
            & (df_transfer["test_band"] == b2)
        ]["circuit_accuracy"]

        # HF->LF: train on b2 (higher freq), test on b1 (lower freq)
        hf_lf = df_transfer[
            (df_transfer["model"] == model)
            & (df_transfer["train_band"] == b2)
            & (df_transfer["test_band"] == b1)
        ]["circuit_accuracy"]

        if len(lf_hf) > 0 and len(hf_lf) > 0:
            rows_asym.append(
                {
                    "model": model,
                    "low_band": b1,
                    "high_band": b2,
                    "freq_distance": r2 - r1,
                    "lf_to_hf": lf_hf.mean(),
                    "hf_to_lf": hf_lf.mean(),
                    "asymmetry": lf_hf.mean() - hf_lf.mean(),
                }
            )

df_asym = pd.DataFrame(rows_asym)
save_analysis(df_asym, "07_transfer_asymmetry.csv")
print(f"Asymmetry data: {len(df_asym)} rows")

# Aggregate: is there a systematic LF->HF vs HF->LF difference?
if not df_asym.empty:
    mean_asym = df_asym["asymmetry"].mean()
    t_stat, p_val = stats.ttest_1samp(df_asym["asymmetry"], 0)
    print(f"\nMean asymmetry (LF->HF minus HF->LF): {mean_asym:.4f}")
    print(f"One-sample t-test: t={t_stat:.3f}, p={p_val:.4e}")
    print(f"Interpretation: {'LF->HF better' if mean_asym > 0 else 'HF->LF better'}")

Asymmetry data: 30 rows

Mean asymmetry (LF->HF minus HF->LF): 0.0880
One-sample t-test: t=5.971, p=1.7258e-06
Interpretation: LF->HF better


In [21]:
if not df_asym.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel 1: LF->HF vs HF->LF scatter
    ax = axes[0]
    for model in MODELS:
        df_m = df_asym[df_asym["model"] == model]
        ax.scatter(
            df_m["lf_to_hf"],
            df_m["hf_to_lf"],
            color=MODEL_COLORS.get(model, "gray"),
            label=model,
            s=50,
            alpha=0.7,
        )

    lims = [0, max(df_asym[["lf_to_hf", "hf_to_lf"]].max()) * 1.1]
    ax.plot(lims, lims, "k--", alpha=0.3, label="Symmetry line")
    ax.set_xlabel("LF \u2192 HF Transfer Accuracy")
    ax.set_ylabel("HF \u2192 LF Transfer Accuracy")
    ax.set_title("Transfer Asymmetry")
    ax.legend(fontsize=8)

    # Panel 2: Asymmetry vs frequency distance
    ax = axes[1]
    for model in MODELS:
        df_m = df_asym[df_asym["model"] == model]
        ax.scatter(
            df_m["freq_distance"],
            df_m["asymmetry"],
            color=MODEL_COLORS.get(model, "gray"),
            label=model,
            s=50,
            alpha=0.7,
        )

    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
    ax.set_xlabel("Frequency Distance")
    ax.set_ylabel("Asymmetry (LF\u2192HF - HF\u2192LF)")
    ax.set_title("Asymmetry vs Frequency Distance")
    ax.legend(fontsize=8)

    fig.suptitle("Cross-Band Transfer Asymmetry", y=1.02)
    fig.tight_layout()
    save_figure(fig, "viz_07_06_transfer_asymmetry.png")

In [22]:
# Link asymmetry to representational differences
# Hypothesis: asymmetry correlates with convergence layer difference (directional)
if not df_conv.empty and not df_asym.empty:
    conv_by_band = (
        df_conv.groupby(["model", "band"])["mean_convergence_layer"]
        .mean()
        .reset_index()
    )

    asym_repr = []
    for _, row in df_asym.iterrows():
        m = row["model"]
        c_low = conv_by_band[
            (conv_by_band["model"] == m) & (conv_by_band["band"] == row["low_band"])
        ]["mean_convergence_layer"]
        c_high = conv_by_band[
            (conv_by_band["model"] == m) & (conv_by_band["band"] == row["high_band"])
        ]["mean_convergence_layer"]

        if len(c_low) > 0 and len(c_high) > 0:
            asym_repr.append(
                {
                    "model": m,
                    "asymmetry": row["asymmetry"],
                    "convergence_diff_directed": c_high.values[0] - c_low.values[0],
                }
            )

    df_asym_repr = pd.DataFrame(asym_repr)
    if len(df_asym_repr) >= 5:
        r, p = stats.pearsonr(
            df_asym_repr["convergence_diff_directed"], df_asym_repr["asymmetry"]
        )
        print(f"Convergence diff (HF-LF) vs transfer asymmetry: r={r:.4f}, p={p:.4e}")
        save_analysis(df_asym_repr, "07_asymmetry_representational_link.csv")

Convergence diff (HF-LF) vs transfer asymmetry: r=-0.5781, p=8.2079e-04


## 8. Summary

Compile all cross-band transfer correlations into a single summary table.

In [23]:
# Collect all correlations
all_corrs = []

# Embedding CKA vs transfer
if "embedding_cka" in df_pairs.columns:
    mask = df_pairs[["embedding_cka", "transfer_mean"]].notna().all(axis=1)
    df_v = df_pairs[mask]
    if len(df_v) >= 5:
        r, p = stats.pearsonr(df_v["embedding_cka"], df_v["transfer_mean"])
        all_corrs.append({"metric": "embedding_cka", "r": r, "p": p, "n": len(df_v)})

# Jaccard vs transfer
mask = df_pairs[["jaccard", "transfer_mean"]].notna().all(axis=1)
df_v = df_pairs[mask]
if len(df_v) >= 5:
    r, p = stats.pearsonr(df_v["jaccard"], df_v["transfer_mean"])
    all_corrs.append({"metric": "jaccard", "r": r, "p": p, "n": len(df_v)})

# Residual CKA cols
for col in ["resid_cka_early", "resid_cka_middle", "resid_cka_final"]:
    if col not in df_pairs.columns:
        continue
    mask = df_pairs[[col, "transfer_mean"]].notna().all(axis=1)
    df_v = df_pairs[mask]
    if len(df_v) >= 5:
        r, p = stats.pearsonr(df_v[col], df_v["transfer_mean"])
        all_corrs.append({"metric": col, "r": r, "p": p, "n": len(df_v)})

# Convergence diff, probe diff
for col in ["convergence_diff", "probe_diff", "freq_distance"]:
    if col not in df_pairs.columns:
        continue
    mask = df_pairs[[col, "transfer_mean"]].notna().all(axis=1)
    df_v = df_pairs[mask]
    if len(df_v) >= 5:
        r, p = stats.pearsonr(df_v[col], df_v["transfer_mean"])
        all_corrs.append({"metric": col, "r": r, "p": p, "n": len(df_v)})

df_summary = pd.DataFrame(all_corrs)
if not df_summary.empty:
    df_summary = df_summary.sort_values("r", key=abs, ascending=False)
    # BH-FDR correction
    from statsmodels.stats.multitest import multipletests

    _, pvals_corrected, _, _ = multipletests(df_summary["p"].values, method="fdr_bh")
    df_summary["p_fdr"] = pvals_corrected
    df_summary["significant"] = df_summary["p_fdr"] < ALPHA

    save_analysis(df_summary, "07_transfer_correlation_summary.csv")
    print("Transfer correlation summary:")
    print(df_summary.to_string(index=False))

Transfer correlation summary:
          metric         r            p  n        p_fdr  significant
 resid_cka_early  0.894997 1.890268e-18 50 1.323188e-17         True
         jaccard -0.829358 1.003923e-13 50 3.513732e-13         True
resid_cka_middle  0.806264 1.598844e-12 50 3.730636e-12         True
   embedding_cka  0.790337 1.317541e-09 40 2.305696e-09         True
 resid_cka_final  0.680493 5.428396e-08 50 7.599754e-08         True
convergence_diff  0.228478 1.105092e-01 50 1.289274e-01        False
   freq_distance -0.119047 5.309366e-01 30 5.309366e-01        False


In [24]:
# Summary bar chart: correlation strength
if not df_summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["coral" if sig else "lightgray" for sig in df_summary["significant"]]
    ax.barh(df_summary["metric"], df_summary["r"], color=colors)
    ax.axvline(x=0, color="gray", linestyle="-", linewidth=0.5)
    ax.set_xlabel("Pearson r with Transfer Accuracy")
    ax.set_title("Correlation Summary: Metrics vs Cross-Band Transfer")

    from matplotlib.patches import Patch

    ax.legend(
        handles=[
            Patch(facecolor="coral", label=f"Significant (FDR < {ALPHA})"),
            Patch(facecolor="lightgray", label="Not significant"),
        ]
    )
    fig.tight_layout()
    save_figure(fig, "viz_07_07_correlation_summary.png")

print("\nNB07 complete.")


NB07 complete.
